[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/tabular-ml-practice/03_tree_models/03_tree_models_solutions.ipynb)

# 03. 결정 트리와 랜덤 포레스트 — 연습 문제 해설

[03_tree_models.ipynb](03_tree_models.ipynb) 끝의 연습 문제 6개에 대한 정답 코드와 해설입니다.
**먼저 직접 시도해본 뒤** 참고하세요.

본문과 같은 순서입니다. **문제 1~2는 1부(택시·회귀), 문제 3~6은 2부(타이타닉·분류)** 범위입니다.

> **읽는 법** — 셀은 위에서부터 순서대로 실행해야 합니다(`Shift + Enter`). 실행 결과는 저장되어
> 있지 않으니 직접 실행해야 표와 그래프가 나타납니다. 맨 위의 **준비 셀들을 먼저 실행한 뒤**
> 원하는 문제로 건너뛰면 됩니다. 해설에 적힌 숫자는 실행하면 나오는 값입니다.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q pandas seaborn matplotlib scikit-learn koreanize-matplotlib

### 준비 셀

아래 셀들은 본문과 같은 준비 코드입니다. **내용을 이해할 필요 없이 그대로 실행**하면 됩니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 그래프에 한글이 깨지지 않도록 폰트를 설정합니다.
# koreanize-matplotlib이 있으면 그걸 쓰고, 없으면 OS에 설치된 한글 폰트를 찾습니다.
try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    import matplotlib.font_manager as fm

    for _name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
        if any(_name == f.name for f in fm.fontManager.ttflist):
            plt.rc("font", family=_name)
            break
plt.rcParams["axes.unicode_minus"] = False  # 한글 폰트에서 마이너스 기호가 깨지는 것 방지

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

본문과 같은 데이터·같은 전처리를 씁니다. **문제 1~2는 `trips`(회귀), 문제 3~6은 `titanic`(분류)** 입니다.

In [ ]:
trips = sns.load_dataset("taxis")
trips["pickup"] = pd.to_datetime(trips["pickup"])
trips["dropoff"] = pd.to_datetime(trips["dropoff"])
trips["duration"] = (trips["dropoff"] - trips["pickup"]).dt.total_seconds() / 60
trips["speed"] = trips["distance"] / (trips["duration"] / 60)
trips["weekday"] = trips["pickup"].dt.dayofweek
trips["hour"] = trips["pickup"].dt.hour

titanic = sns.load_dataset("titanic")

print("trips  :", trips.shape)
print("titanic:", titanic.shape)

02번에서 만든 전처리 함수 두 개입니다.

In [ ]:
def prepare_trips(raw):
    """[회귀] 택시 데이터를 학습 가능한 형태로 만든다. 02_preprocessing에서 단계별로 만든 코드."""
    df = raw[(raw["duration"] > 0) & (raw["speed"] < 60)].copy()   # 이상치 제거
    df = df.drop(columns=["pickup", "dropoff",                     # 시각 자체는 weekday/hour로 대체
                          "pickup_zone", "dropoff_zone",           # 범주가 200개 이상이라 제외
                          "speed",                                 # duration으로 계산한 값 → 정답 누출
                          "total"])                                # fare+tip+tolls의 합 → 중복
    df = df.dropna()                                               # 결측치 행 제거
    df = pd.get_dummies(df, columns=["color", "payment",
                                     "pickup_borough", "dropoff_borough"],
                        drop_first=True)                           # 범주형 → 0/1
    X = df.drop(columns="duration")
    y = df["duration"]
    return X, y


def prepare_titanic(raw):
    """[분류] 타이타닉 데이터를 학습 가능한 형태로 만든다. 02_preprocessing에서 단계별로 만든 코드."""
    q1, q3 = raw["fare"].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

    df = raw[(raw["fare"] >= lower) & (raw["fare"] <= upper)].copy()  # 이상치 제거
    df = df.drop(columns=["alive",                                   # survived와 같은 정보 → 정답 누출
                          "class", "embark_town",                    # pclass/embarked와 중복
                          "deck",                                    # 결측치가 77%
                          "adult_male"])                             # who와 중복
    df = df.dropna()
    df = pd.get_dummies(df, columns=["sex", "embarked", "who"], drop_first=True)
    X = df.drop(columns="survived")
    y = df["survived"]
    return X, y

회귀 쪽은 본문 1부에서 찾아낸 누출 컬럼(`fare`·`tip`·`tolls`)을 **제거한 상태**로 시작합니다.

In [ ]:
from sklearn.model_selection import train_test_split

X_reg, y_reg = prepare_trips(trips)
X_clf, y_clf = prepare_titanic(titanic)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)
# 본문에서 찾아낸 데이터 누출 컬럼을 제거한 상태로 시작합니다
leaky = ["fare", "tip", "tolls"]
X_train = X_train.drop(columns=leaky)
X_valid = X_valid.drop(columns=leaky)

Xc_train, Xc_valid, yc_train, yc_valid = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=RANDOM_STATE, stratify=y_clf
)

print(f"[회귀] 학습 {X_train.shape}  검증 {X_valid.shape}")
print(f"[분류] 학습 {Xc_train.shape}  검증 {Xc_valid.shape}")

---

# 1부 — 택시 (회귀)

## 문제 1. `GridSearchCV` vs `RandomizedSearchCV`

In [ ]:
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, 20, None],
    "min_samples_leaf": [1, 2, 5],
    "min_samples_split": [2, 5, 10],
}

t0 = time.time()
gs = GridSearchCV(RandomForestRegressor(random_state=RANDOM_STATE), param_grid,
                  cv=3, scoring="neg_mean_absolute_error", n_jobs=-1).fit(X_train, y_train)
t_grid = time.time() - t0

t0 = time.time()
rs = RandomizedSearchCV(RandomForestRegressor(random_state=RANDOM_STATE), param_grid,
                        n_iter=10, cv=3, scoring="neg_mean_absolute_error",
                        n_jobs=-1, random_state=RANDOM_STATE).fit(X_train, y_train)
t_rand = time.time() - t0

pd.DataFrame({
    "시험한 조합": [len(gs.cv_results_["params"]), len(rs.cv_results_["params"])],
    "소요 시간(초)": [t_grid, t_rand],
    "교차검증 MAE": [-gs.best_score_, -rs.best_score_],
    "검증 MAE": [mean_absolute_error(y_valid, gs.predict(X_valid)),
                 mean_absolute_error(y_valid, rs.predict(X_valid))],
}, index=["GridSearchCV", "RandomizedSearchCV"]).round(4)

두 방식이 고른 조합과 소요 시간 차이를 확인합니다.

In [ ]:
print("GridSearchCV      최적:", gs.best_params_)
print("RandomizedSearchCV 최적:", rs.best_params_)
print()
print(f"시간 비율: RandomizedSearchCV가 {t_grid / t_rand:.1f}배 빠름")
print(f"교차검증 MAE 차이: {abs(gs.best_score_ - rs.best_score_):.4f}분")

**해설**

| | 조합 | 시간 | 교차검증 MAE | 검증 MAE |
|---|---|---|---|---|
| `GridSearchCV` | **108개** | 55.5초 | **3.4603** | 3.3872 |
| `RandomizedSearchCV` | 10개 | **6.8초** | 3.4686 | **3.3851** |

**시간은 8배 차이인데 성능은 사실상 같습니다.** 교차검증 MAE 차이가 0.008분(0.5초)이고,
검증 데이터에서는 오히려 `RandomizedSearchCV`가 근소하게 나았습니다.

**왜 10개만 봐도 충분한가**

하이퍼파라미터 대부분은 **성능에 거의 영향을 주지 않습니다.** 위 격자에서 실제로 중요한 것은
`max_depth`(둘 다 10을 골랐습니다) 정도이고, `n_estimators`가 100이든 300이든,
`min_samples_split`이 2든 10이든 결과가 비슷합니다.

격자 탐색은 **중요하지 않은 파라미터의 모든 값에 대해서도 성실하게 전부 시험**합니다.
`n_estimators`를 하나 더 추가하면 조합이 36개 늘어나는데, 얻는 것은 거의 없습니다.

**언제 무엇을 쓸까**

| | `GridSearchCV` | `RandomizedSearchCV` |
|---|---|---|
| 후보가 적을 때 (< 50조합) | ✅ | 굳이 |
| 후보가 많을 때 | 시간 폭발 | ✅ |
| 연속값 범위 탐색 | 불가능 (목록만) | ✅ **확률 분포 지정 가능** |
| 재현성 | 완전 | `random_state` 필요 |

마지막 항목이 `RandomizedSearchCV`의 진짜 강점입니다. 목록 대신 **분포**를 줄 수 있습니다.

```python
from scipy.stats import randint, uniform

param_dist = {
    "n_estimators": randint(50, 500),        # 50~500 사이 아무 정수
    "max_depth": randint(3, 30),
    "min_samples_leaf": randint(1, 20),
    "max_features": uniform(0.3, 0.7),       # 0.3~1.0 사이 실수
}
```

격자로는 `[100, 200, 300]` 같은 이산 목록만 줄 수 있어서, **최적값이 250이라면 영영 찾지 못합니다.**

**실무 순서**는 보통 이렇습니다.

1. `RandomizedSearchCV`로 넓은 범위를 성기게 훑어 유망한 구간을 찾는다
2. 그 근처를 `GridSearchCV`로 촘촘히 다시 훑는다

## 문제 2. 잔차 분석 — 어디서 많이 틀리는가

In [ ]:
best_rf = RandomForestRegressor(**gs.best_params_, random_state=RANDOM_STATE, n_jobs=-1)
best_rf.fit(X_train, y_train)

pred = best_rf.predict(X_valid)

analysis = X_valid.copy()
analysis["실제"] = y_valid
analysis["예측"] = pred
analysis["잔차"] = y_valid - pred            # 양수면 과소예측, 음수면 과대예측
analysis["절대오차"] = analysis["잔차"].abs()

print(f"전체 MAE: {analysis['절대오차'].mean():.3f} 분")
print(f"잔차 평균: {analysis['잔차'].mean():.3f} 분")

먼저 **이동 거리 구간별**로 오차를 봅니다.

In [ ]:
# ① 이동 거리별
analysis["거리 구간"] = pd.cut(analysis["distance"], [0, 1, 2, 5, 10, 50],
                               labels=["~1마일", "1-2", "2-5", "5-10", "10마일+"])

by_distance = analysis.groupby("거리 구간").agg(
    건수=("잔차", "size"),
    평균_절대오차=("절대오차", "mean"),
    평균_잔차=("잔차", "mean"),
).round(3)
by_distance

다음은 **실제 이동 시간 구간별**입니다. 여기서 문제가 가장 뚜렷하게 드러납니다.

In [ ]:
# ② 실제 이동 시간별 — 여기서 문제가 가장 뚜렷하게 보입니다
analysis["시간 구간"] = pd.cut(analysis["실제"], [0, 5, 10, 20, 40, 120],
                               labels=["~5분", "5-10분", "10-20분", "20-40분", "40분+"])

by_duration = analysis.groupby("시간 구간").agg(
    건수=("잔차", "size"),
    평균_절대오차=("절대오차", "mean"),
    평균_잔차=("잔차", "mean"),
).round(3)
by_duration

**시간대별**로도 나눠 봅니다.

In [ ]:
# ③ 시간대별
analysis["시간대"] = pd.cut(analysis["hour"], [-1, 5, 10, 16, 20, 23],
                            labels=["새벽(0-5)", "오전(6-10)", "낮(11-16)", "저녁(17-20)", "밤(21-23)"])

analysis.groupby("시간대").agg(
    건수=("잔차", "size"),
    평균_절대오차=("절대오차", "mean"),
    평균_잔차=("잔차", "mean"),
).round(3)

표로 본 것을 그래프로 확인합니다. 왼쪽 상자들이 빨간 선(잔차 0)의 위아래 어느 쪽으로 치우쳐 있는지 보세요.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=analysis, x="시간 구간", y="잔차", ax=axes[0])
axes[0].axhline(0, color="red", linestyle="--", linewidth=1)
axes[0].set_title("실제 이동 시간별 잔차")

axes[1].scatter(analysis["실제"], analysis["예측"], alpha=0.3, s=10)
lims = [0, analysis["실제"].max()]
axes[1].plot(lims, lims, "r--", linewidth=1)
axes[1].set_xlabel("실제 (분)")
axes[1].set_ylabel("예측 (분)")
axes[1].set_title("실제 vs 예측")

plt.tight_layout()
plt.show()

**해설 — 모델은 "평균으로 끌어당기는" 편향을 가지고 있습니다**

### 발견 ①: 오차는 거리·시간에 비례해 커진다

| 거리 | 건수 | 평균 절대오차 |
|---|---|---|
| ~1마일 | 357 | **1.77분** |
| 2-5마일 | 302 | 3.81분 |
| 10마일+ | 77 | **7.87분** |

당연해 보이지만 중요한 확인입니다. **절대오차만 보면 "긴 운행이 어렵다"로 끝납니다.**
진짜 문제는 다음에 있습니다.

### 발견 ②: 짧은 운행은 과대예측, 긴 운행은 과소예측 (핵심)

`잔차 = 실제 - 예측`이므로 **양수면 실제보다 짧게 예측**(과소예측)한 것입니다.

| 실제 시간 | 건수 | 평균 잔차 | 해석 |
|---|---|---|---|
| ~5분 | 193 | **-1.54** | 실제보다 **길게** 예측 |
| 5-10분 | 408 | -1.12 | 길게 예측 |
| 10-20분 | 399 | -0.34 | 거의 정확 |
| 20-40분 | 217 | +1.37 | 짧게 예측 |
| **40분+** | 51 | **+9.53** | **9.5분이나 짧게 예측** |

**40분 넘는 운행을 평균 9.5분이나 짧게 예측합니다.** 50분 걸릴 길을 40분이라고 알려주는
모델입니다. 공항 가는 승객에게는 치명적입니다.

**원인 두 가지**

1. **트리 모델은 잎에 남은 데이터의 평균을 예측합니다.** 잎 안에 20분짜리와 60분짜리가
   섞여 있으면 둘 다 그 평균으로 예측됩니다. 구조적으로 **극단값 쪽으로 못 갑니다.**
   그래서 학습 데이터의 최댓값을 절대 넘는 예측을 하지 못합니다
2. **긴 운행 데이터가 적습니다.** 40분 이상이 검증 데이터 1,268건 중 51건(4%)뿐입니다.
   학습 데이터에서도 비슷한 비율이라, 모델이 이 구간을 배울 기회가 적었습니다

### 개선 아이디어

**① 타깃을 로그 변환한다**

```python
y_log = np.log1p(y_train)              # log(1 + y)
model.fit(X_train, y_log)
pred = np.expm1(model.predict(X_valid))  # 되돌리기
```

오른쪽으로 치우친 분포를 좌우대칭에 가깝게 만들어, **긴 운행의 오차가 상대적으로 더 크게
반영**되도록 합니다. 01번에서 본 "오른쪽으로 늘어진 분포"에 대한 표준적인 대응입니다.

**② 오차 자체를 다르게 정의한다**

절대 오차(분) 대신 **비율 오차**를 쓰면, 5분짜리를 1분 틀리는 것과 50분짜리를 10분 틀리는 것을
같게 취급합니다. 도착 시간 안내에서는 이쪽이 더 자연스러운 기준일 수 있습니다.

**③ 긴 운행에 가중치를 준다**

`fit(X, y, sample_weight=...)`로 긴 운행의 비중을 높입니다.

**④ 부족한 정보를 채운다**

가장 근본적인 해법입니다. 40분 넘는 운행은 대부분 공항 노선인데, 현재 피처에는
**목적지가 공항인지 알려주는 정보가 없습니다**(zone을 버렸으니까요).
02번 연습문제 3번에서 다룬 "상위 N개 지역 + Other" 방식으로 zone을 되살리면
공항 노선을 구분할 수 있습니다.

**⑤ 부스팅 계열 모델을 쓴다**

`GradientBoostingRegressor`, XGBoost, LightGBM은 **이전 모델이 틀린 부분을 다음 모델이
집중적으로 학습**합니다. 랜덤 포레스트가 놓치는 소수 구간을 더 잘 잡아내는 경우가 많습니다.

> **잔차 분석은 "MAE 3.4분"이라는 숫자 하나가 감추고 있던 것을 드러냅니다.**
> 전체 평균은 괜찮아 보여도, 정작 예측이 중요한 구간에서 체계적으로 틀리고 있을 수 있습니다.
> 모델을 평가할 때 지표 하나로 끝내지 말고 **어디서 틀리는지** 반드시 확인하세요.

---

# 2부 — 타이타닉 (분류)

## 문제 3. 분류에서의 과적합 곡선

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

depths = [1, 2, 3, 4, 5, 7, 10, 15, 20, None]
rows = []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE).fit(Xc_train, yc_train)
    rows.append({
        "max_depth": "제한 없음" if d is None else d,
        "실제 깊이": m.get_depth(),
        "학습 정확도": accuracy_score(yc_train, m.predict(Xc_train)),
        "검증 정확도": accuracy_score(yc_valid, m.predict(Xc_valid)),
    })

res = pd.DataFrame(rows)
res.round(4)

표를 곡선으로 그려 본문 1부의 회귀 그래프와 모양을 비교합니다.
(깊이 16 이후로는 트리가 더 자라지 않아 X축을 16까지만 씁니다.)

In [ ]:
plot_x = [1, 2, 3, 4, 5, 7, 10, 15, 16, 16]

plt.figure(figsize=(9, 5))
plt.plot(plot_x, res["학습 정확도"], "o-", label="학습 데이터")
plt.plot(plot_x, res["검증 정확도"], "s-", label="검증 데이터")
plt.axvline(3, color="gray", linestyle="--", linewidth=1)
plt.text(3.2, 0.80, "최적 지점 (깊이 3)", color="gray")
plt.xlabel("트리 깊이 (max_depth)")
plt.ylabel("정확도")
plt.title("분류에서의 과적합 — 깊이 3에서 이미 정점")
plt.legend()
plt.show()

**해설 — 모양은 같지만 정점이 훨씬 빠릅니다**

| 깊이 | 학습 정확도 | 검증 정확도 |
|---|---|---|
| 1 | 0.7669 | 0.7717 |
| **3** | 0.8322 | **0.8207 ← 최고** |
| 5 | 0.8648 | 0.7880 |
| 10 | 0.9650 | 0.7337 |
| 16(제한 없음) | **0.9930** | 0.7283 |

회귀에서는 깊이 10이 최적이었는데 **분류는 깊이 3**입니다. 그 이유는 **데이터 양**입니다.

| | 학습 데이터 | 최적 깊이 |
|---|---|---|
| 회귀 (택시) | **5,068건** | 10 |
| 분류 (타이타닉) | **429건** | **3** |

깊이 d인 트리는 최대 2^d개의 잎을 가집니다. 깊이 10이면 잎이 최대 1,024개인데,
데이터가 429건뿐이면 **잎 하나에 한 명도 안 들어갑니다.** 규칙이 아니라 명부가 되는 셈입니다.

**데이터가 적을수록 모델을 단순하게 유지해야 한다**는 것이 일반 원칙입니다.
본문에서 `GridSearchCV`가 `max_depth=10`을 골랐는데, 그때는 `min_samples_leaf=5`,
`min_samples_split=20`이 함께 걸려 있어서 **실질적으로는 훨씬 얕은 트리**가 만들어졌습니다.
깊이만으로 복잡도를 판단하면 안 되는 이유입니다.

또 하나 눈여겨볼 점은 **깊이 16 이후로 변화가 없다**는 것입니다. `max_depth=20`이나
`None`이나 실제 깊이는 16입니다. 더 나눌 데이터가 남지 않아 트리가 스스로 멈춘 것입니다.

## 문제 4. OOB 점수

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

rf_oob = RandomForestClassifier(oob_score=True, random_state=RANDOM_STATE, n_jobs=-1)
rf_oob.fit(Xc_train, yc_train)

cv_scores = cross_val_score(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    Xc_train, yc_train, cv=5, scoring="accuracy"
)

print(f"OOB 점수        : {rf_oob.oob_score_:.4f}   (학습 한 번으로 계산)")
print(f"5겹 교차 검증    : {cv_scores.mean():.4f}   (학습 5번 필요)")
print(f"  각 fold       : {cv_scores.round(4)}")
print(f"검증 데이터 점수 : {accuracy_score(yc_valid, rf_oob.predict(Xc_valid)):.4f}")

"각 트리가 데이터의 63%만 본다"는 말이 어디서 나왔는지 시뮬레이션으로 확인해봅시다.

In [ ]:
# 부트스트랩에서 한 번도 뽑히지 않을 확률을 시뮬레이션으로 확인
n = 429
rng = np.random.default_rng(0)

not_selected = []
for _ in range(200):
    sample = rng.integers(0, n, size=n)        # 복원 추출로 n개 뽑기
    not_selected.append(1 - len(np.unique(sample)) / n)

print(f"뽑히지 않은 데이터 비율(시뮬레이션 평균): {np.mean(not_selected):.4f}")
print(f"이론값 1/e                              : {1 / np.e:.4f}")
print(f"즉 각 트리는 데이터의 약 {(1 - 1 / np.e) * 100:.1f}% 만 학습에 사용")

**해설 — OOB(Out-Of-Bag)란**

랜덤 포레스트의 각 트리는 **복원 추출(bootstrap)** 로 뽑은 데이터로 학습합니다.
n개에서 n개를 복원 추출하면 **중복이 생기고, 어떤 데이터는 한 번도 안 뽑힙니다.**

특정 데이터 하나가 n번의 추첨에서 매번 안 뽑힐 확률은

```
(1 - 1/n)^n  →  1/e ≈ 0.368        (n이 커질 때)
```

즉 **각 트리는 데이터의 약 63.2%만 보고 학습하고, 나머지 36.8%는 그 트리 입장에서
"본 적 없는 데이터"** 입니다. 이 남은 데이터를 **out-of-bag 샘플**이라고 합니다.

**OOB 점수**는 각 데이터에 대해 "그 데이터를 학습에 쓰지 않은 트리들"만 모아 예측하게 하고,
그 정확도를 잰 것입니다. 별도의 검증 데이터를 떼어내지 않고도 일반화 성능을 추정할 수 있습니다.

| 방법 | 점수 | 학습 횟수 |
|---|---|---|
| OOB | 0.7949 | **1번** |
| 5겹 교차 검증 | 0.8111 | 5번 |
| 별도 검증 데이터 | 0.7554 | 1번 |

**OOB의 장점**

- **학습 한 번으로 끝납니다.** 교차 검증은 5번 학습해야 합니다
- 검증용으로 데이터를 떼어낼 필요가 없어, **데이터가 적을 때 특히 유용**합니다

**한계**

- **부트스트랩을 쓰는 모델에서만** 가능합니다 (`bootstrap=False`면 계산 안 됨)
- 각 데이터가 평균 100 × 0.368 ≈ 37개 트리로만 평가되므로, 전체 100개 트리를 쓰는 최종 모델보다
  **약간 비관적으로 나오는 경향**이 있습니다
- 점수 하나만 나와서 **교차 검증처럼 분산(fold별 편차)을 알 수 없습니다**

세 값이 0.755 ~ 0.811로 흩어져 있는데, 검증 데이터가 184건뿐이라 그렇습니다.
**데이터가 적을 때는 어떤 추정치든 폭넓게 봐야 합니다.**

## 문제 5. 재현율을 높이는 두 가지 방법

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1).fit(Xc_train, yc_train)
proba = rf.predict_proba(Xc_valid)[:, 1]

rows = []
for th in [0.5, 0.4, 0.3, 0.2]:
    pred = (proba >= th).astype(int)
    rows.append({
        "방법": f"임계값 {th}",
        "정확도": accuracy_score(yc_valid, pred),
        "정밀도": precision_score(yc_valid, pred),
        "재현율": recall_score(yc_valid, pred),
        "F1": f1_score(yc_valid, pred),
    })

# class_weight="balanced"
rf_bal = RandomForestClassifier(class_weight="balanced",
                                random_state=RANDOM_STATE, n_jobs=-1).fit(Xc_train, yc_train)
pred_bal = rf_bal.predict(Xc_valid)
rows.append({
    "방법": "class_weight=balanced",
    "정확도": accuracy_score(yc_valid, pred_bal),
    "정밀도": precision_score(yc_valid, pred_bal),
    "재현율": recall_score(yc_valid, pred_bal),
    "F1": f1_score(yc_valid, pred_bal),
})

pd.DataFrame(rows).set_index("방법").round(4)

임계값을 낮추면 **놓친 생존자(FN)와 거짓 경보(FP)가 실제로 어떻게 맞바뀌는지** 봅니다.

In [ ]:
# 임계값을 낮추면 혼동 행렬이 어떻게 바뀌는지
for th in [0.5, 0.3]:
    pred = (proba >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(yc_valid, pred).ravel()
    print(f"임계값 {th}:  놓친 생존자(FN) {fn:2d}명   거짓 경보(FP) {fp:2d}명")

`class_weight="balanced"`가 내부에서 계산하는 가중치를 직접 구해봅니다.

In [ ]:
# class_weight가 하는 일: 소수 클래스에 가중치를 준다
counts = yc_train.value_counts()
n_classes = 2
print("학습 데이터 클래스 분포:", counts.to_dict())
print()
for cls, cnt in counts.items():
    w = len(yc_train) / (n_classes * cnt)
    print(f"  클래스 {cls}의 가중치 = {len(yc_train)} / ({n_classes} × {cnt}) = {w:.4f}")
print()
print(f"불균형 비율: {counts[0] / counts[1]:.2f} : 1")

**해설**

| 방법 | 정확도 | 정밀도 | 재현율 | F1 |
|---|---|---|---|---|
| 임계값 0.5 (기본) | 0.7609 | 0.6400 | 0.7385 | 0.6857 |
| 임계값 0.4 | 0.7609 | 0.6329 | 0.7692 | **0.6944** |
| **임계값 0.3** | 0.7391 | 0.5955 | **0.8154** | 0.6883 |
| 임계값 0.2 | 0.6957 | 0.5437 | **0.8615** | 0.6667 |
| `class_weight="balanced"` | 0.7554 | 0.6429 | 0.6923 | 0.6667 |

**① 임계값 조정 — 의도한 대로 작동합니다**

0.5 → 0.3으로 낮추면 재현율이 0.7385 → 0.8154로 오릅니다. 놓친 생존자가 17명에서 12명으로
줄어드는 대신, 거짓 경보가 27명에서 32명으로 늘어납니다. **정확히 예상한 trade-off**입니다.

임계값을 더 낮출수록 재현율은 계속 오르지만(0.2에서 0.8615), 정밀도가 0.54까지 떨어집니다.
극단적으로 임계값을 0으로 하면 전부 "생존"이라 예측해서 재현율 100%, 정밀도는 클래스 비율이 됩니다.

**② `class_weight="balanced"` — 여기서는 효과가 없었습니다**

재현율이 오히려 **떨어졌습니다**(0.7385 → 0.6923). 예상과 반대입니다. 이유가 있습니다.

`balanced`는 클래스 가중치를 `전체 / (클래스 수 × 해당 클래스 개수)`로 설정합니다.
여기서는 사망 277명 / 생존 152명으로 **불균형이 1.8 : 1에 불과합니다.** 가중치가
0.774 vs 1.411 정도라 모델 동작이 크게 바뀌지 않고, 그 작은 변화가 우연히
검증 데이터에서 나쁘게 나온 것입니다(184건이라 2~3명 차이면 뒤집힙니다).

**`class_weight`는 불균형이 심할 때 효과적입니다.** 양성이 1%인 사기 탐지라면
가중치가 50배 차이 나므로 모델이 확실히 달라집니다.

**결론 — 어느 쪽을 쓸까**

- **임계값 조정이 더 직접적이고 예측 가능합니다.** 학습을 다시 할 필요도 없고,
  운영 중에 정책만 바꿔 조절할 수 있습니다
- `class_weight`는 **학습 단계에서** 소수 클래스를 더 중요하게 취급하게 만듭니다.
  불균형이 심할 때 먼저 시도해볼 만합니다
- 둘을 함께 쓸 수도 있습니다

> **F1이 가장 높은 것은 임계값 0.4(0.6944)** 입니다. "정밀도와 재현율의 균형"이 목표라면
> 이 지점이 최적입니다. 하지만 재현율이 절대적으로 중요한 문제라면 F1이 조금 낮아도
> 임계값 0.2를 고르는 것이 맞습니다. **지표는 목적에 따라 고르는 것이지, F1이 항상 정답은 아닙니다.**

## 문제 6. 왜 이 코드의 결과를 믿을 수 없는가

**문제로 주어진 코드**

```python
gs = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5)
gs.fit(X_clf, y_clf)                       # 전체 데이터로 탐색
print("최종 성능:", gs.best_score_)         # 이 점수를 최종 성능으로 보고
```

In [ ]:
# 실제로 얼마나 낙관적으로 나오는지 확인
gs_all = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    {"max_depth": [3, 5, 7, 10, None], "min_samples_leaf": [1, 3, 5, 10]},
    cv=5, scoring="accuracy", n_jobs=-1,
).fit(X_clf, y_clf)          # ❌ 전체 데이터

gs_train = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    {"max_depth": [3, 5, 7, 10, None], "min_samples_leaf": [1, 3, 5, 10]},
    cv=5, scoring="accuracy", n_jobs=-1,
).fit(Xc_train, yc_train)    # ✅ 학습 데이터만

print("① 전체 데이터로 탐색")
print(f"   best_score_ (보고한 성능): {gs_all.best_score_:.4f}")
print()
print("② 학습 데이터로만 탐색")
print(f"   best_score_             : {gs_train.best_score_:.4f}")
print(f"   따로 떼어둔 검증 데이터   : {accuracy_score(yc_valid, gs_train.predict(Xc_valid)):.4f}")

조합을 많이 시험할수록 1등 점수가 부풀려지는지, 상위 k개의 평균과 비교해 확인합니다.

In [ ]:
# 조합을 많이 시험할수록 best_score_가 부풀려지는 정도
sizes = [1, 5, 10, 20]
all_scores = np.sort(gs_all.cv_results_["mean_test_score"])[::-1]

for k in sizes:
    print(f"상위 {k:2d}개 조합의 평균 점수: {all_scores[:k].mean():.4f}   "
          f"(1등만: {all_scores[0]:.4f})")

**해설 — 문제가 두 겹입니다**

### ① 검증 데이터를 남겨두지 않았다

`gs.fit(X_clf, y_clf)`는 **가진 데이터 전부**를 탐색에 씁니다. 교차 검증이 내부적으로
나누기는 하지만, **모든 데이터가 어느 시점에는 하이퍼파라미터 선택에 관여**했습니다.

즉 **완전히 처음 보는 데이터에서의 성능을 확인할 방법이 없습니다.**

### ② `best_score_`는 구조적으로 낙관적이다

이게 더 미묘한 문제입니다. `best_score_`는 20개 조합 중 **가장 높은 점수**입니다.
각 점수에는 진짜 실력과 우연이 섞여 있는데, **최댓값을 고르면 "운이 좋았던 것"이
선택될 가능성이 큽니다.** 이것을 **선택 편향(selection bias)** 이라고 합니다.

②번 실험이 그 격차를 보여줍니다. 학습 데이터로만 탐색했을 때

| | 점수 |
|---|---|
| `best_score_` (탐색 중 가장 좋았던 교차 검증 점수) | **0.8251** |
| 따로 떼어둔 검증 데이터에서의 실제 점수 | **0.7772** |

**4.8%p 차이**입니다. `best_score_`를 최종 성능으로 보고했다면 실제보다 훨씬 후하게
평가한 셈입니다.

같은 경향이 조합별 점수에서도 보입니다. 1등 조합(0.8108)은 상위 20개 평균(0.8042)보다
높습니다. 차이는 작지만 **방향이 일정하고, 조합을 많이 시험할수록 커집니다.**
100개 조합을 시험하면 그중 하나는 우연히 좋은 점수를 받기 마련입니다.

> ①과 ②의 `best_score_`를 직접 비교하는 것(0.8108 vs 0.8251)은 의미가 없습니다.
> 쓴 데이터 양이 613건과 429건으로 다르기 때문입니다. **문제의 핵심은 ①이 점수가
> 높다는 것이 아니라, ①에는 성능을 검증할 데이터가 아예 남아 있지 않다는 것**입니다.

### 올바른 방법

```python
# 1. 먼저 검증(또는 테스트) 데이터를 떼어낸다 — 이후 절대 건드리지 않는다
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

# 2. 학습 데이터 안에서만 교차 검증으로 탐색
gs = GridSearchCV(model, param_grid, cv=5).fit(X_train, y_train)

# 3. 최종 성능은 떼어둔 데이터로 딱 한 번 측정
final_score = gs.score(X_test, y_test)
```

**데이터를 세 갈래로 나누는 것이 정석입니다.**

| 용도 | 이름 | 역할 |
|---|---|---|
| 학습 | train | 모델 파라미터를 맞춘다 |
| 검증 | validation | **하이퍼파라미터를 고른다** (교차 검증이 이 역할) |
| 테스트 | test | **최종 성능을 딱 한 번 측정한다** |

**테스트 데이터를 보고 모델을 고치는 순간 그것은 더 이상 테스트 데이터가 아닙니다.**
"테스트 점수가 낮으니 파라미터를 바꿔볼까"를 반복하면, 결국 테스트 데이터에 과적합됩니다.

> 데이터가 적어서 세 갈래로 나누기 아깝다면 **중첩 교차 검증(nested CV)** 을 씁니다.
> 바깥 루프가 성능 측정, 안쪽 루프가 하이퍼파라미터 탐색을 담당합니다.
> `cross_val_score(GridSearchCV(...), X, y, cv=5)` 형태로 간단히 쓸 수 있지만,
> 학습 횟수가 곱해지므로 시간이 많이 듭니다.

---

다음 노트북: [04_dnn_keras.ipynb](../04_dnn_keras/04_dnn_keras.ipynb)